In [1]:
from loqs.backends import QSimQuantumState, PyGSTiNoiseModel
from loqs.core import QuantumProgram

from loqs.codepacks import d3_5q_code

import pygsti

In [2]:
code_5q = d3_5q_code.create_qec_code()

In [3]:
code_5q.instructions.keys()

dict_keys(['Non-FT Minus Prep', 'X', 'Z', 'H', 'Non-FT Raw Z Measure'])

In [4]:
# Define a pyGSTi processor spec and noise model
# This is the only pyGSTi required here,
# and could eventually be traded out for something else
qubits = ["A0", "A1"] + [f"D{i+2}" for i in range(5)]
gate_names = ["Gxpi2", "Gypi2", "Gzpi2", "Gh", "Gcnot", "Gcphase"]

# TODO: Currently Iz does not need to be set here
# This is because QSimQuantumState actually does not try to pull the rep
# Otherwise, this would result in a KeyError in PyGSTiNoiseModel.get_reps(),
# since it technically should be provided
pspec = pygsti.processors.QubitProcessorSpec(
    len(qubits), 
    gate_names=gate_names,
    qubit_labels=qubits,
    availability={k: "all-combinations" for k in gate_names}
)

ideal_model_pygsti = pygsti.models.create_crosstalk_free_model(pspec)

ideal_model = PyGSTiNoiseModel(ideal_model_pygsti)


In [5]:
# Logical quantum program information
patch_types = {"5Q": code_5q}

# Stack
# The "Init *" instructions will be generated by the QuantumProgram based on the values
# of state_type and patch_types below
stack = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}), # Autogenerated from state_tyep
    ("Init Patch 5Q", None, ("L0", qubits)), # Autogenerated from patch_types. Note that 5Q here must be a key
    ("Non-FT Minus Prep", "L0"),
    ("H", "L0"),
    ("Non-FT Raw Z Measure", "L0")
]

program = QuantumProgram(
    stack,
    default_noise_model=ideal_model,
    state_type=QSimQuantumState,
    patch_types=patch_types,
    name="Test program"
)

In [6]:
# You can do a dry-run, which skips all the apply_fn
# parts of most instructions but otherwise runs the
# whole program through to ensure that args/kwargs
# are available as desired
# I decided to go this route rather than checking
# input/outputs will match everywhere with the hope
# that this will be more efficient. You can do one dry-run
# once taking the actual codepaths through the workflow,
# and then avoid all checks for that program
program.run(dry_run=True)

# If you were to look at the output history,
# you would see "DRY_RUN" for entries like the state
# and measurement_outcomes.
# TBD how I will do these for feedforward operations,
# perhaps just provide a default to use for processing in that case

Executing program run 0
Working on frame 1
Working on frame 2
Working on frame 3
Working on frame 4
Working on frame 5
Working on frame 6
Working on frame 7
Dry run completed successfully!


In [8]:
# And now we can run it for real!
program.run()

Executing program run 1
Working on frame 1
Working on frame 2
Working on frame 3
Working on frame 4
Working on frame 5
Working on frame 6
Working on frame 7


In [9]:
# We can get a pretty verbose output by looking at the history
print(program.run_histories[1])

# Note that the state for everything but the final frame
# is listed as EXPIRED.
# This is my way of indicating that this is a single object that we
# are not storing copies of at each frame. Which keys expire is a
# settable thing in the History object, although an advanced usage

History with 8 items:
  Frame Adding InstructionStack from new QuantumProgram Test program (1 objects):
    'stack': 
      InstructionStack with 5 items:
        InstructionLabel(Init State,None,(7,),{qubit_labels: ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']})
        InstructionLabel(Init Patch 5Q,None,('L0', ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']),{})
        InstructionLabel(Non-FT Minus Prep,L0,(),{model: Physical pyGSTi noise model})
        InstructionLabel(H,L0,(),{model: Physical pyGSTi noise model,patch_label: L0,stack: InstructionStack with 1 items:
          InstructionLabel(Non-FT Raw Z Measure,L0,(),{model: Physical pyGSTi noise model})})
        InstructionLabel(Non-FT Raw Z Measure,L0,(),{model: Physical pyGSTi noise model})

  Frame QSimQuantumState state builder result (5 objects):
    'state': EXPIRED
    'instruction': 
      Instruction QSimQuantumState state builder
        Input parameter spec:
          InputSpec with 4 items:
            InputParam(0,['label

In [10]:
# A good example of the dry run is looking at the last frame for both runs
print("Dry run final frame")
print(program.run_histories[0][-1])

print("\nReal run final frame")
print("Dry run final frame")
print(program.run_histories[1][-1])

# You'll note that things like the stack and patches were done faithfully
# but things like state and measurement_outcomes are not computed
# in the dry run

Dry run final frame
Frame Z-basis measurement for information-containing physical qubits result (7 objects):
  'state': 
    DRY_RUN
  'measurement_outcomes': 
    DRY_RUN
  'instruction': 
    Instruction Z-basis measurement for information-containing physical qubits
      Input parameter spec:
        InputSpec with 6 items:
          InputParam(0,['label'],model,None,None)
          InputParam(1,['default'],circuit,None,None)
          InputParam(2,['history'],state,-1,state)
          InputParam(3,['default'],include_outcomes,None,None)
          InputParam(4,['default'],inplace,None,None)
          InputParam(5,['default'],reset_mcms,None,None)
      Output frame keys: ['state', 'measurement_outcomes']
      Defaults:
        include_outcomes: True
        inplace: True
        reset_mcms: False
        circuit: Physical pyGSTi circuit:
          Qubit A0 ---|  |---
          Qubit A1 ---|  |---
          Qubit D5 ---|Iz|---
          Qubit D2 ---|  |---
          Qubit D4 ---|Iz|